![Logo Javeriana](https://images.seeklogo.com/logo-png/11/1/pontificia-universidad-javeriana-logo-png_seeklogo-110703.png)

## Nombres: 
- Gabriel Jaramillo Cuberos (Id: 20529022)
- Samuel Beltrán Martínez (Id: 20524509)
- David Vargas (Id: 20485686)
- Juan Felipe Gómez López (Id: 20553416)
- Sara Pulgarín (Id: 20491129)
- Juan Camilo Delgado (Id: 20506996)

### Primera entrega del proyecto

*Materia:* Procesamiento de Datos (1255)

*Tema:* Consultoría Nueva York.

*Descripción:* En este cuaderno se hace una integración de los resultados de los demás cuadernos.

*Nombre del fichero:* 04_Integracion_Preguntas.ipynb

# Cuaderno 4 - Integración, viabilidad y preguntas de negocio


**Primera entrega: Entendimiento del negocio y de los datos**

**Año objetivo: 2018**

El equipo actúa como consultor para elaborar un diagnóstico histórico de la ciudad de **Nueva York**.
El objetivo es describir arrestos, siniestros y condiciones socioeconómicas, preparar indicadores comparables
y documentar limitaciones para una futura priorización. La primera entrega no presenta respuestas definitivas
a las ocho preguntas ni un plan de intervención cerrado. Un arresto no prueba un delito ni identifica una persona única.

Los CSV, ubicados en la carpeta de datasets del clúster, se leen con una función compartida del archivo *proyecto.py*. Se ejecuta Spark en el cluster del taller, con un máximo de dos núcleos por aplicación (esto se hace para no ocupar todos los recursos del clúster).

**Objetivos del cuaderno de integración:**

- Reunit los resúmenes de arrestos, pobreza y educación.
- Incorporar la población, usando el web scraper, para calcular arrestos por cada 100.000 habitantes y comparar distritos de distinto tamaño.
- Comprobar qué indicadores están disponibles y cuáles faltan.
- Guardar una tabla integrada que servirá como punto de partida para responder las preguntas de negocio en la entrega final.

In [1]:
# Comprobar kernel

import sys
import pyspark

print("Python:", sys.version)
print("Ejecutable:", sys.executable)
print("PySpark:", pyspark.__version__)

Python: 3.11.13 (main, Jun  1 2026, 00:00:00) [GCC 11.5.0 20240719 (Red Hat 11.5.0-14)]
Ejecutable: /home/estudiante/jupyter-spark-env/bin/python
PySpark: 4.2.0


## 1. Configuración del cluster

La configuración compartida reproduce los recursos y el Python del taller.

In [3]:
# Se inicia la sesión spark.

from Utils import proyecto as p
from Utils.proyecto import ANIO, DATA_DIR, RESULTADOS
spark = p.iniciar_spark("Entrega1_Integracion")
from pyspark.sql import functions as F
from IPython.display import display, Markdown
print("Directorio de entrada:", DATA_DIR) # Imprime el directorio de donde llegan los datos.
print("Directorio de resultados:", RESULTADOS) # Imprime el directorio a donde van los resultados.

Python: 3.11.13 | PySpark: 4.2.0 | Spark: 4.2.0
Ejecutable: /home/estudiante/jupyter-spark-env/bin/python
Aplicación: app-20260928135516-0024 | Máximo de núcleos: 2
Los recursos físicos del cluster deben documentarse desde la interfaz del master.
Directorio de entrada: /opt/cluster/Proyecto/Entrega_1/datasets
Directorio de resultados: /opt/cluster/Proyecto/Entrega_1/resultados


## 2. Comunicación entre los cuadernos

El 01 guarda `arrestos_resumen.json`, el 02 guarda `vehiculos_resumen.json`, el 03 guarda `pobreza_resumen.json` y el bono guarda `poblacion_nyc_2018.csv` en `/opt/cluster/Proyecto/Entrega_1/resultados/`. Este cuaderno lee esos archivos y obtiene los resúmenes que incluyen período, esquema y estado de disponibilidad.
Los JSON son pequeños y se leen en la máquina del notebook para luego convertirse en DataFrames de Spark.

Los 3 cuadernos anteriores debieron ser previamente ejecutados para el correcto funcionamiento de este, pues requiere de sus resultados.

In [4]:
# Cada llamada a cargar_resumen devuelve un DataFrame de Spark con los datos resumidos y un diccionario con información del resumen

# Se carga el resumen del cuaderno de arrestos
arrestos, meta_arrestos = p.cargar_resumen(
    spark,
    "arrestos_resumen.json"
)

# Se carga el resumen del cuaderno de accidentes viales
vehiculos, meta_vehiculos = p.cargar_resumen(
    spark,
    "vehiculos_resumen.json"
)

# Se carga el resumen del cuaderno de pobreza y educación
pobreza, meta_pobreza = p.cargar_resumen(
    spark,
    "pobreza_resumen.json"
)

# Se agrupan los metadatos de los resúmenes
metadatos = [
    meta_arrestos,
    meta_vehiculos,
    meta_pobreza
]

# Se crean una fila por resumen para mostrar información
filas_informativas = []

for meta in metadatos:
    fila = {
        "fuente": meta["fuente"],
        "anio_objetivo": meta["anio_objetivo"],
        "estado": meta["estado"]
    }

    filas_informativas.append(fila)

# Se muestran resultados
p.tabla(filas_informativas)

fuente,anio_objetivo,estado
Arrestos_NY.csv,2018,disponible
Accidentes_Viales_NY.csv,2018,sin_geografia_ni_victimas
Pobreza_NY.csv,2018,disponible


## 3. Población del bono

En este bloque se carga el dataset resultante del web scraper. El objetivo es integrarlo con los resultados de los demás cuadernos para calcular tasas de las diferentes métricas obtenidas.

In [5]:
import pandas as pd
from pathlib import Path

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    LongType,
)

# Se genera una estructura para construir dataframe

schema_poblacion = StructType([
    StructField("anio", IntegerType(), False),
    StructField("borough", StringType(), False),
    StructField("poblacion", LongType(), False),
])


# Se invoca el CSV con la información de la población

ruta_poblacion = (
    Path(RESULTADOS) / "poblacion_nyc_2018.csv"
)

poblacion_disponible = ruta_poblacion.exists()

# Se establece una lista para los registros
registros = []


# Se lee el archivo
if poblacion_disponible:

    # Leer el CSV como una tabla de pandas
    datos = pd.read_csv(ruta_poblacion)

    # Se verifica existencia de columnas
    columnas_necesarias = {"anio", "borough", "poblacion"}

    if not columnas_necesarias.issubset(datos.columns):
        raise ValueError(
            "El CSV debe contener: anio, borough y poblacion."
        )

    # Se verifica que haya exactamente cinco filas y que correspondan a los cinco distritos esperados
    if (
        len(datos) != 5
        or set(datos["borough"]) != set(p.BOROUGHS)
    ):
        raise ValueError(
            "El CSV debe contener los cinco distritos de NYC sin duplicados."
        )

    # Se convierte el año y la población a números
    for columna in ["anio", "poblacion"]:
        datos[columna] = pd.to_numeric(
            datos[columna],
            errors="coerce",
        )

    # Se verifica el año de los registros
    if not datos["anio"].eq(ANIO).all():
        raise ValueError(
            f"Todos los registros deben corresponder a {ANIO}."
        )

    # Se verifica la población
    if (
        datos["poblacion"].isna().any()
        or not datos["poblacion"].gt(0).all()
        or not datos["poblacion"].mod(1).eq(0).all()
    ):
        raise ValueError(
            "La población debe contener números enteros positivos."
        )

    # Se genera una lista de diccionarios para Spark
    for fila in datos.itertuples(index=False):
        registro = {
            "anio": int(fila.anio),
            "borough": fila.borough,
            "poblacion": int(fila.poblacion),
        }

        registros.append(registro)

    print("Población cargada desde:", ruta_poblacion)

else:
    print(
        "No se encontró el CSV de población."
    )


# Se crea el DataFrame de Spark

poblacion = spark.createDataFrame(
    registros,
    schema=schema_poblacion,
)


# Se muestra la población cargada

poblacion.orderBy("borough").show(truncate=False)

Población cargada desde: /opt/cluster/Proyecto/Entrega_1/resultados/poblacion_nyc_2018.csv


[Stage 0:>                                                          (0 + 2) / 2]

+----+-------------+---------+
|anio|borough      |poblacion|
+----+-------------+---------+
|2018|Bronx        |1432000  |
|2018|Brooklyn     |2583000  |
|2018|Manhattan    |1629000  |
|2018|Queens       |2279000  |
|2018|Staten Island|476000   |
+----+-------------+---------+



## 4. Unión por año y distrito

En este bloque se parte de los 5 distritos y hace los siguientes procedimientos:
- Revisa las tablas
- Une las tablas por distrito y año
- Calcula la tasa de arrestos
- Muestra qué información hay
- Guarda los resultados

Adicionalmente, se hace una validación de los registros parap evitar duplicaciones y prevenir la repetición de la población. Además, se hace un `left join` por si falta algún resumen de alguna métrica de algún distrito. 

El dataset de vehículos utilizado no tiene distrito ni cantidades de personas lesionadas o fallecidas. Por lo tanto, no se calculan estos indicadores ni se integran territorialmente.

In [7]:
# Se revisan las tablas antes de unirlas
tablas = [
    ("arrestos", arrestos),
    ("pobreza", pobreza),
    ("poblacion", poblacion),
]

for nombre, tabla in tablas:

    # Se hace un conteo de filas. Debe haber máximo una por año y distrito
    conteo_por_distrito = (
        tabla
        .groupBy("anio", "borough")
        .count()
    )

    # Se buscan combinaciones que aparecen más de una vez
    duplicados = conteo_por_distrito.filter(
        F.col("count") > 1
    )

    if duplicados.count() > 0:
        raise ValueError(
            f"Hay años y distritos repetidos en {nombre}. "
            "Revisa esa tabla antes de unirla."
        )

    # Se buscan filas con año diferente al del proyecto y con distrito vacío o fuera de los cinco distritos esperados
    claves_invalidas = tabla.filter(
        (F.col("anio") != ANIO)
        | F.col("anio").isNull()
        | F.col("borough").isNull()
        | (~F.col("borough").isin(p.BOROUGHS))
    )

    if claves_invalidas.count() > 0:
        raise ValueError(
            f"Hay años o distritos no válidos en {nombre}."
        )

# Se cra una tabla base con los 5 distritos

# Se prepara una fila por distrito
filas_base = []

for distrito in p.BOROUGHS:
    filas_base.append((ANIO, distrito))

base = spark.createDataFrame(
    filas_base,
    schema="anio int, borough string"
)

# Se une la población, pobreza y arrestos

# Se crea una clave para relacionar las tablas
claves = ["anio", "borough"]

# La operación left mantiene todos los distritos de la tabla de la izquierda y, si no encuentra datos en la otra tabla, deja valores nulos
integrada = base.join(
    poblacion,
    claves,
    "left"
)

integrada = integrada.join(
    pobreza,
    claves,
    "left"
)

integrada = integrada.join(
    arrestos,
    claves,
    "left"
)

# Se calculan arrestos por cada 100000 personas con la fórmula cantidad de arrestos / población × 100000
integrada = integrada.withColumn(
    "arrestos_por_100mil",
    F.when(
        F.col("poblacion") > 0,
        100000.0 * F.col("arrestos") / F.col("poblacion")
    )
)

# Se comprueba que las uniones tienen los cinco distritos
if integrada.count() != 5:
    raise ValueError(
        "La tabla integrada debe tener exactamente cinco filas, una por distrito."
    )

# Se seleccionan los datos que permiten interpretar la tasa
tabla_tasas = integrada.select(
    "anio",
    "borough",
    "arrestos",
    "poblacion",

    # Se redondea la tasa a dos decimales
    F.round("arrestos_por_100mil", 2).alias(
        "arrestos_por_100mil"
    )
).orderBy("borough")

# Se imprime la tabla
p.tabla(p.filas(tabla_tasas))

# Se muestra información disponible
disponibilidad = integrada.select(
    "anio",
    "borough",

    F.col("poblacion")
    .isNotNull()
    .alias("hay_poblacion"),

    F.col("pobreza_pct")
    .isNotNull()
    .alias("hay_pobreza"),

    F.col("arrestos")
    .isNotNull()
    .alias("hay_arrestos_2018"),

    F.lit(False)
    .alias("hay_siniestros_por_distrito")
)

filas_disponibilidad = p.filas(
    disponibilidad.orderBy("borough")
)

p.tabla(filas_disponibilidad)

# Guardar resultados

# Se guarda la tabla que indica qué información hay
ruta_disponibilidad = p.guardar_json(
    "04_disponibilidad.json",
    filas_disponibilidad
)

# Se hace una tabla integrada
filas_integradas = p.filas(
    integrada.orderBy("borough")
)

reporte_integrado = {
    "anio": ANIO,
    "datos": filas_integradas,
    "nota": (
        "Los null son información no disponible. No son ceros. "
    )
}

ruta_integrada = p.guardar_json(
    "04_base_territorial_preliminar.json",
    reporte_integrado
)

print("Disponibilidad guardada en:", ruta_disponibilidad)
print("Tabla integrada guardada en:", ruta_integrada)

anio,borough,arrestos,poblacion,arrestos_por_100mil
2018,Bronx,215,1432000,15.01
2018,Brooklyn,237,2583000,9.18
2018,Manhattan,209,1629000,12.83
2018,Queens,206,2279000,9.04
2018,Staten Island,50,476000,10.5


anio,borough,hay_poblacion,hay_pobreza,hay_arrestos_2018,hay_siniestros_por_distrito
2018,Bronx,True,True,True,False
2018,Brooklyn,True,True,True,False
2018,Manhattan,True,True,True,False
2018,Queens,True,True,True,False
2018,Staten Island,True,True,True,False


Disponibilidad guardada en: /opt/cluster/Proyecto/Entrega_1/resultados/04_disponibilidad.json
Tabla integrada guardada en: /opt/cluster/Proyecto/Entrega_1/resultados/04_base_territorial_preliminar.json


## Cierre

Los JSON y PNG quedan guardados. 

In [ ]:
# Cierra sesión y limpia cache

spark.catalog.clearCache()
spark.stop()